<h1>Calculate Normalized mean-squeared error at different iterations</h1>

In [20]:
import sys
import os

# Add the path to the `simulated_recons/` folder to the Python path
sys.path.append(os.path.abspath("../../.."))
import livedifferences

from ptypy import io
from ptypy.utils import rmphaseramp
import numpy as np
import re
import glob
from skimage.registration import phase_cross_correlation
from scipy.ndimage import shift
import matplotlib.pyplot as plt
import time
%matplotlib widget
plt.ion()


<h2>Load data</h2>

In [21]:
ol_cases = {'10px': {'offline_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step10px_1e+10_poisTRUE_spiral_00/simg_startframe400__fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1840.ptyr', 
                     'realtime_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step10px_1e+10_poisTRUE_spiral_00/simg_startframe1____fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1840.ptyr', 
                     'samplename': '_spiralstep10px_1e10_apert-pr-update'}, 
            '19px': {'offline_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step19px_1e+10_poisTRUE_spiral_00/simg_startframe315__fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1710.ptyr', 
                     'realtime_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step19px_1e+10_poisTRUE_spiral_00/simg_startframe1____fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1710.ptyr', 
                     'samplename': '_spiralstep19px_1e10_apert-pr-update'}, 
            '27px': {'offline_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step27px_1e+10_poisTRUE_spiral_00/simg_startframe315__fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1660.ptyr', 
                     'realtime_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step27px_1e+10_poisTRUE_spiral_00/simg_startframe1____fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1660.ptyr', 
                     'samplename': '_spiralstep27px_1e10_apert-pr-update'}, 
            '35px': {'offline_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step35px_1e+10_poisTRUE_spiral_00/simg_startframe315__fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1570.ptyr', 
                     'realtime_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step35px_1e+10_poisTRUE_spiral_00/simg_startframe1____fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1570.ptyr', 
                     'samplename': '_spiralstep35px_1e10_apert-pr-update'}, 
            '40px': {'offline_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step40px_1e+10_poisTRUE_spiral_00/simg_startframe315__fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1530.ptyr', 
                     'realtime_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step40px_1e+10_poisTRUE_spiral_00/simg_startframe1____fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1530.ptyr', 
                     'samplename': '_spiralstep40px_1e10_apert-pr-update'} }

LD10 = livedifferences.LoadData(offline_fname=ol_cases['10px']['offline_fname'], realtime_fname=ol_cases['10px']['realtime_fname'], samplename=ol_cases['10px']['samplename'])
LD10.load()
LD10.process_data() # Gives obj1_abs, obj2_abs, obj1_abslog, obj2_abslog, obj1_phase, obj2_phase, w1, w2, obj1_ramp, ramp1, obj2_ramp, ramp2, obj1_phaseramp, obj2_phaseramp, obj1_phaseramp_unwrapped, obj2_phaseramp_unwrapped, obj1_rmramp_abs, obj2_rmramp_abs, obj1_rmramp_abslog, obj2_rmramp_abslog, diff_rmramp_abslog, diff_abslog_od, diff_phase, diff_phaseramp, diff_phaseramp_unwrapped, diff_phaseramp_unwrapped_ramp, shift1, error, phasediff
LD10.load_GT() # Gives fname3, fname_pr, pr, obj, obj_abs, obj_abslog, obj_phase, w, obj_ramp, ramp, obj_rmramp_abs, obj_rmramp_abslog, obj_phaseramp, obj_phaseramp_unwrapped
LD10.add_padding() # Gives sh, sh1, padrow, padcol, obj1_pad, obj2_pad, obj1_abs_pad, obj2_abs_pad, obj1_abslog_pad, obj2_abslog_pad, obj1_phase_pad, obj2_phase_pad, obj1_ramp_pad, obj2_ramp_pad, obj1_rmramp_abs_pad, obj2_rmramp_abs_pad, obj1_rmramp_abslog_pad, obj2_rmramp_abslog_pad, obj1_phaseramp_pad, obj2_phaseramp_pad, obj1_phaseramp_unwrapped_pad, obj2_phaseramp_unwrapped_pad

marg = ((min(LD10.sh) - max(LD10.sh1)) // 2) + int(max(LD10.sh1) // 3.15)
obj_GT = np.array(LD10.obj_ramp[marg:-marg,marg:-marg].copy())



LD40 = livedifferences.LoadData(offline_fname=ol_cases['40px']['offline_fname'], realtime_fname=ol_cases['40px']['realtime_fname'], samplename=ol_cases['40px']['samplename'])
LD40.load()
LD40.process_data() # Gives obj1_abs, obj2_abs, obj1_abslog, obj2_abslog, obj1_phase, obj2_phase, w1, w2, obj1_ramp, ramp1, obj2_ramp, ramp2, obj1_phaseramp, obj2_phaseramp, obj1_phaseramp_unwrapped, obj2_phaseramp_unwrapped, obj1_rmramp_abs, obj2_rmramp_abs, obj1_rmramp_abslog, obj2_rmramp_abslog, diff_rmramp_abslog, diff_abslog_od, diff_phase, diff_phaseramp, diff_phaseramp_unwrapped, diff_phaseramp_unwrapped_ramp, shift1, error, phasediff
LD40.load_GT() # Gives fname3, fname_pr, pr, obj, obj_abs, obj_abslog, obj_phase, w, obj_ramp, ramp, obj_rmramp_abs, obj_rmramp_abslog, obj_phaseramp, obj_phaseramp_unwrapped
LD40.add_padding() # Gives sh, sh1, padrow, padcol, obj1_pad, obj2_pad, obj1_abs_pad, obj2_abs_pad, obj1_abslog_pad, obj2_abslog_pad, obj1_phase_pad, obj2_phase_pad, obj1_ramp_pad, obj2_ramp_pad, obj1_rmramp_abs_pad, obj2_rmramp_abs_pad, obj1_rmramp_abslog_pad, obj2_rmramp_abslog_pad, obj1_phaseramp_pad, obj2_phaseramp_pad, obj1_phaseramp_unwrapped_pad, obj2_phaseramp_unwrapped_pad


<h2>Calculate</h2>

In [22]:
# Normalized mean-squeared error

def NMSE(obj1_, obj2_, obj_GT, printoutput=True):
    """Calculated as in eqn (8) and (9) in Maiden, A. M. & Rodenburg, J. M. An improved ptychographical phase retrieval algorithm for diffractive imaging.
Ultramicroscopy 109, 1256–1262 (2009)."""
    W1 = LD.rampweight(obj1_, scale=100)#
    obj1_ = rmphaseramp(obj1_, W1)
    obj1_ = rmphaseramp(obj1_, W1)
    obj1_ = rmphaseramp(obj1_, W1)
    obj1_ = rmphaseramp(obj1_, W1)
    obj1_ = rmphaseramp(obj1_, W1)
    obj2_ = rmphaseramp(obj2_, W1)
    obj2_ = rmphaseramp(obj2_, W1)
    obj2_ = rmphaseramp(obj2_, W1)
    obj2_ = rmphaseramp(obj2_, W1)
    obj2_ = rmphaseramp(obj2_, W1)
    obj_GT = rmphaseramp(obj_GT, W1)
    obj_GT = rmphaseramp(obj_GT, W1)
    obj_GT = rmphaseramp(obj_GT, W1)
    obj_GT = rmphaseramp(obj_GT, W1)
    obj_GT = rmphaseramp(obj_GT, W1)


    gamma1 = ( np.sum( obj_GT*np.conjugate(obj1_) ) ) / (np.sum( np.abs(obj1_)**2 ))
    gamma2 = ( np.sum( obj_GT*np.conjugate(obj2_) ) ) / (np.sum( np.abs(obj2_)**2 ))
    gamma12 = ( np.sum( obj1_*np.conjugate(obj2_) ) ) / (np.sum( np.abs(obj2_)**2 ))
    gamma21 = ( np.sum( obj2_*np.conjugate(obj1_) ) ) / (np.sum( np.abs(obj1_)**2 ))  # Just to ensure the order doesn't matter

    #print(np.abs(gamma1), np.mean(np.abs(obj_GT)/np.abs(obj1_)))
    E1 = ( np.sum( np.abs(obj_GT-gamma1*obj1_)**2 ) ) / ( np.sum( np.abs(obj_GT)**2 ) )
    E2 = ( np.sum( np.abs(obj_GT-gamma2*obj2_)**2 ) ) / ( np.sum( np.abs(obj_GT)**2 ) )
    E12 = ( np.sum( np.abs(obj1_-gamma12*obj2_)**2 ) ) / ( np.sum( np.abs(obj1_)**2 ) )
    E21 = ( np.sum( np.abs(obj2_-gamma21*obj1_)**2 ) ) / ( np.sum( np.abs(obj2_)**2 ) )  # Just to ensure the order doesn't matter
    
    if printoutput:
        print(f'NMSE(obj_GT, obj1_): {E1:.2e}')
        print(f'NMSE(obj_GT, obj2_): {E2:.2e}')
        print(f'NMSE(obj1_,  obj2_): {E12:.2e}')
        print(f'NMSE(obj2_,  obj1_): {E21:.2e}')


    #### Peak signal to noise ratio
    MSE1 = np.mean(np.abs(obj_GT - obj1_)**2)
    MSE2 = np.mean(np.abs(obj_GT - obj2_)**2)
    MSE12 = np.mean(np.abs(obj1_ - obj2_)**2)

    max_val = np.max(np.abs(obj_GT))
    max_val1 = np.max(np.abs(obj1_))

    PSNR1 = 10 * np.log10(max_val**2 / MSE1)
    PSNR2 = 10 * np.log10(max_val**2 / MSE2)
    PSNR12 = 10 * np.log10(max_val1**2 / MSE12)
    
    if printoutput:
        print('PSNR: ', PSNR1, PSNR2, PSNR12)
    return E1, E2, E12


In [51]:
# Load saved data if this cell has been run before, otherwise do the NMSE calculations.
NMSE_iter_fname = "NMSE_iter_data_upsamp100.npz"
if os.path.exists(NMSE_iter_fname):
    with np.load(NMSE_iter_fname) as data:
        Err1 = data['Err1']
        Err2 = data['Err2']
        Err12 = data['Err12']
        its = data['its']
else:
    LD = LD40
    #marg = ((min(LD10.sh) - max(LD10.sh1)) // 2) + int(max(LD10.sh1) // 3.15)
    #obj_GT = np.array(LD10.obj_ramp[marg:-marg,marg:-marg].copy())
    Err1 = []
    Err2 = []
    Err12 = []
    its = []
    t0 = time.time()
    t1 = []
    t2 = []
    t3 = []
    t4 = []
    t5 = []
    t6 = []
    t7 = []
    t1 = []
    obj2_alliter = []#np.zeros((165,165,153)) # 153 is the nr of iteration files saved
    obj2_allramps = []
    obj2_allramppads = []
    shifts = []
    obj2_allramppads_shifted = []
    sz = 100 # half the windowsize that should be used for determining translational shift w.r.t. GT
    k=-1
    for it in range(10,1540, 10):
        k+=1
        offline_fname = ol_cases['40px']['offline_fname'][:-9] + str(it).rjust(4,'0') + ".ptyr"
        realtime_fname = ol_cases['40px']['realtime_fname'][:-9] + str(it).rjust(4,'0') + ".ptyr"
        t1.append(time.time())
        LD40 = livedifferences.LoadData(offline_fname=offline_fname, realtime_fname=realtime_fname, samplename=ol_cases['40px']['samplename']) # 719 ns ± 0.659 ns
        t2.append(time.time())
        LD40.load() # 128 ms ± 205 µs

        #process
        LD40.w1 = LD40.rampweight(LD40.obj1, scale=10)#100) #5) #np.abs(rampweight(obj1, scale=3)-1)
        LD40.w2 = LD40.rampweight(LD40.obj2, scale=10)#100) #5) #np.abs(rampweight(obj2, scale=3)-1)
        LD40.obj1_ramp = rmphaseramp(LD40.obj1, LD40.w1)
        LD40.obj2_ramp = rmphaseramp(LD40.obj2, LD40.w2)
        obj2_allramps.append(LD40.obj2_ramp)

        #pad
        LD40.sh = LD10.obj.shape
        LD40.sh1 = LD40.obj1.shape
        LD40.sh2 = LD40.obj2.shape
        LD40.padrow1 = (LD40.sh[0]-LD40.sh1[0])//2
        LD40.padcol1 = (LD40.sh[1]-LD40.sh1[1])//2
        LD40.obj1_ramp_pad = np.pad(LD40.obj1_ramp, ((LD40.padrow1,LD40.sh[0]-LD40.padrow1-LD40.sh1[0]),(LD40.padcol1,LD40.sh[1]-LD40.padcol1-LD40.sh1[1])))
        LD40.padrow2 = (LD40.sh[0]-LD40.sh2[0])//2
        LD40.padcol2 = (LD40.sh[1]-LD40.sh2[1])//2
        LD40.obj2_ramp_pad = np.pad(LD40.obj2_ramp, ((LD40.padrow2,LD40.sh[0]-LD40.padrow2-LD40.sh2[0]),(LD40.padcol2,LD40.sh[1]-LD40.padcol2-LD40.sh2[1])))
        obj2_allramppads.append(LD40.obj2_ramp_pad)

        #shift
        shift2GT = phase_cross_correlation(LD10.obj_ramp[LD40.sh[1]//2-sz:LD40.sh[1]//2+sz, LD40.sh[0]//2-sz:LD40.sh[0]//2+sz], LD40.obj2_ramp_pad[LD40.sh[1]//2-sz:LD40.sh[1]//2+sz, LD40.sh[0]//2-sz:LD40.sh[0]//2+sz], upsample_factor=100, return_error=False)
        shifts.append(shift2GT)
        obj2_ramp_pad_shifted = shift(LD40.obj2_ramp_pad.copy(), shift2GT, mode="constant", cval=0+0j)
        obj2_allramppads_shifted.append(obj2_ramp_pad_shifted)



        """t3.append(time.time())
        LD40.process_data() # 1.91 s ± 13.6 ms # Gives obj1_abs, obj2_abs, obj1_abslog, obj2_abslog, obj1_phase, obj2_phase, w1, w2, obj1_ramp, ramp1, obj2_ramp, ramp2, obj1_phaseramp, obj2_phaseramp, obj1_phaseramp_unwrapped, obj2_phaseramp_unwrapped, obj1_rmramp_abs, obj2_rmramp_abs, obj1_rmramp_abslog, obj2_rmramp_abslog, diff_rmramp_abslog, diff_abslog_od, diff_phase, diff_phaseramp, diff_phaseramp_unwrapped, diff_phaseramp_unwrapped_ramp, shift1, error, phasediff
        t4.append(time.time())
        #LD40.load_GT() # 998 ms ± 10.4 ms # Gives fname3, fname_pr, pr, obj, obj_abs, obj_abslog, obj_phase, w, obj_ramp, ramp, obj_rmramp_abs, obj_rmramp_abslog, obj_phaseramp, obj_phaseramp_unwrapped
        #t5.append(time.time())
        LD40.add_padding() # 47.3 ms ± 544 µs # Gives sh, sh1, padrow, padcol, obj1_pad, obj2_pad, obj1_abs_pad, obj2_abs_pad, obj1_abslog_pad, obj2_abslog_pad, obj1_phase_pad, obj2_phase_pad, obj1_ramp_pad, obj2_ramp_pad, obj1_rmramp_abs_pad, obj2_rmramp_abs_pad, obj1_rmramp_abslog_pad, obj2_rmramp_abslog_pad, obj1_phaseramp_pad, obj2_phaseramp_pad, obj1_phaseramp_unwrapped_pad, obj2_phaseramp_unwrapped_pad
        t6.append(time.time())
        """;
        #print(f'it: {it}, marg: {marg}, LD40.sh: {LD40.sh}, LD40.sh1: {LD40.sh1}, LD40.padrow2: {LD40.padrow2}, LD40.padcol2: {LD40.padcol2}, LD40.obj2_ramp_pad.shape: {LD40.obj2_ramp_pad.shape}')

        obj1_ = np.array(LD40.obj1_ramp_pad[marg:-marg,marg:-marg].copy())
        obj2_ = np.array(obj2_ramp_pad_shifted[marg:-marg,marg:-marg].copy())
        obj2_alliter.append(obj2_)#obj2_alliter[:,:,k] = obj2_
        print(f'it: {it}, marg: {marg}, sh1: {LD40.sh1}, sh2: {LD40.sh2}, obj1_ramp_pad.shape: {LD40.obj1_ramp_pad.shape}, obj2_ramp_pad.shape: {LD40.obj2_ramp_pad.shape}, obj1_.shape: {obj1_.shape}, obj2_.shape: {obj2_.shape}')

        E1, E2, E12 = NMSE(obj1_, obj2_, obj_GT, printoutput=False)
        t7.append(time.time())
        Err1.append(E1)
        Err2.append(E2)
        Err12.append(E12)
        its.append(it)

    t8 = time.time()
    print('time: ', t8-t0)

    # save this data to file if it is not done already
    ####if not os.path.exists("NMSE_iter_data.npz"):
    np.savez(NMSE_iter_fname, Err1=Err1, Err2=Err2, Err12=Err12, its=its)
    print('saved data')

it: 10, marg: 588, sh1: (1016, 1036), sh2: (535, 545), obj1_ramp_pad.shape: (1341, 1341), obj2_ramp_pad.shape: (1341, 1341), obj1_.shape: (165, 165), obj2_.shape: (165, 165)
it: 20, marg: 588, sh1: (1016, 1036), sh2: (535, 545), obj1_ramp_pad.shape: (1341, 1341), obj2_ramp_pad.shape: (1341, 1341), obj1_.shape: (165, 165), obj2_.shape: (165, 165)
it: 30, marg: 588, sh1: (1016, 1036), sh2: (535, 545), obj1_ramp_pad.shape: (1341, 1341), obj2_ramp_pad.shape: (1341, 1341), obj1_.shape: (165, 165), obj2_.shape: (165, 165)
it: 40, marg: 588, sh1: (1016, 1036), sh2: (535, 545), obj1_ramp_pad.shape: (1341, 1341), obj2_ramp_pad.shape: (1341, 1341), obj1_.shape: (165, 165), obj2_.shape: (165, 165)
it: 50, marg: 588, sh1: (1016, 1036), sh2: (535, 545), obj1_ramp_pad.shape: (1341, 1341), obj2_ramp_pad.shape: (1341, 1341), obj1_.shape: (165, 165), obj2_.shape: (165, 165)
it: 60, marg: 588, sh1: (1016, 1036), sh2: (535, 545), obj1_ramp_pad.shape: (1341, 1341), obj2_ramp_pad.shape: (1341, 1341), obj1_

In [42]:
# Get iterations where new frames have been loaded

path40 = LD40.fname2.rsplit('/',2)[0]

frames = []
it = []
    
# Load nr. of frames loaded at each iteration.
fname = glob.glob(path40 + '/backtrace-summary*')[0]
with open(fname, 'r') as f:
    fpi_str = f.read()
# Extract the numbers in the file. Every other entry in this list corresponds to nr 
# of frames that have been loaded and to nr of iterations that have been performed.
fpi_data_flattened = [int(s) for s in re.findall(r'\b\d+\b', fpi_str)] 
frames = fpi_data_flattened[::2]
it = fpi_data_flattened[1::2]
    
# Add last data point manually since it's not written to file.
it.append(it[-1]+1)
frames.append(315)

# Remove duplicates/ get all unique nr of frames
nrframes = list(dict.fromkeys(frames))
newload = [] # iteration numbers where a new load has happened
for nr in nrframes:
    newload.append(frames.index(nr))


In [52]:
# Plot
ncol=1
nrow=1
params = {'legend.fontsize': 13, # 20,
         'axes.labelsize': 17, # 25.2,
         'axes.titlesize': 17, # 25.2,
         'xtick.labelsize': 13, # 20,
         'ytick.labelsize': 13} # 20}
plt.rcParams.update(params)
fig, ax = plt.subplots(nrow, ncol, figsize=(9,4))#(4*ncol, 4*nrow))
fig.subplots_adjust(left=0.1, bottom=0.15, right=0.95, top=0.95, wspace=0.40, hspace=0.2)
ax.plot(its, (Err1), c='#6B60B6') # change yaxis to log scale, put maybe markings where new frames are loaded
ax.plot(its, (Err2), c='#6FB087')
ax.plot(its, (Err12), c='#BA5D90') # 9D4172
ax.set_yscale("log")
#plt.setp(ax[2], title='NMSE')
plt.legend(['Offline vs. GT', 'Real-time vs. GT', 'Real-time vs. Offline'], loc='upper right')
for nl in newload:
    ax.axvline(x=nl, ls=':', lw=1, c='#7f7f7f')
ax.set_xlabel('Iterations')
ax.set_ylabel('NMSE')
plt.show()
fig.suptitle('upsample: 100', y=1)

savefigs=False
if savefigs:
    fig.savefig(LD40.fname2.rsplit('/', 4)[0] + f'/NMSE_vs_iteration.png', dpi=400)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

<h2>Extra</h2>

In [40]:
# testing: plotting object at different iterations
check_iterations = [30,40]
for check_iteration in check_iterations:
    ind = list(its).index(check_iteration)

    ncol=2
    nrow=1
    """params = {'legend.fontsize': 13, # 20,
             'axes.labelsize': 17, # 25.2,
             'axes.titlesize': 17, # 25.2,
             'xtick.labelsize': 13, # 20,
             'ytick.labelsize': 13} # 20}
    plt.rcParams.update(params)""";
    fig, ax = plt.subplots(nrow, ncol, figsize=(9,4))#(4*ncol, 4*nrow))
    fig.subplots_adjust(left=0.1, bottom=0.05, right=0.95, top=0.95, wspace=0.40, hspace=0.2)
    ax[0].imshow(np.abs(obj2_alliter[ind]) - np.abs(obj_GT))
    ax[1].imshow(np.angle(obj2_alliter[ind]) - np.angle(obj_GT))
    fig.suptitle(f'Iteration nr: {check_iteration}', fontsize=16)

30 2


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

40 3


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

1

In [19]:
i = 9
its[i],Err1[i], Err2[i]

(100, 4.104975493332375e-07, 4.6798590899804423e-07)

In [20]:
i = -1
its[i],Err1[i], Err2[i]

(1530, 3.4113540138842473e-07, 3.60194155876692e-07)

In [7]:
# Plot some of the objects
ncol=5
nrow=1
# the full reconstructed objects
fig, ax = plt.subplots(nrow, ncol, figsize=(3*ncol, 3*nrow))
plt.tight_layout()
for k in range(ncol):
    ax[k].imshow(np.angle(obj2_allramps[k]), extent=[.0, obj2_allramps[k].shape[0], .0, obj2_allramps[k].shape[1]])
    ax[k].axvline(x=obj2_allramps[k].shape[1]/2, c='k', lw=0.5)
    ax[k].axhline(y=obj2_allramps[k].shape[0]/2, c='k', lw=0.5)
    plt.setp(ax[k], title=str(obj2_allramps[k].shape))

# after adding padding to them
ncol=6
fig, ax = plt.subplots(nrow, ncol, figsize=(3*ncol, 3*nrow))
plt.tight_layout()
for k in range(ncol-1):
    ax[k].imshow(np.angle(obj2_allramppads[k]), extent=[.0, obj2_allramppads[k].shape[0], .0, obj2_allramppads[k].shape[1]])
    ax[k].axvline(x=obj2_allramppads[k].shape[1]/2, c='k', lw=0.5)
    ax[k].axhline(y=obj2_allramppads[k].shape[0]/2, c='k', lw=0.5)
    plt.setp(ax[k], title=str(obj2_allramppads[k].shape))
ax[5].imshow(np.angle(obj_GT), extent=[.0, obj_GT.shape[0], .0, obj_GT.shape[1]])
ax[5].axvline(x=obj_GT.shape[1]/2, c='r', lw=0.5)
ax[5].axhline(y=obj_GT.shape[0]/2, c='r', lw=0.5)
plt.setp(ax[5], title=str(obj_GT.shape))

# after adding the shift
fig, ax = plt.subplots(nrow, ncol, figsize=(3*ncol, 3*nrow))
plt.tight_layout()
for k in range(ncol-1):
    ax[k].imshow(np.angle(obj2_allramppads_shifted[k]), extent=[.0, obj2_allramppads_shifted[k].shape[0], .0, obj2_allramppads_shifted[k].shape[1]])
    ax[k].axvline(x=obj2_allramppads_shifted[k].shape[1]/2, c='k', lw=0.5)
    ax[k].axhline(y=obj2_allramppads_shifted[k].shape[0]/2, c='k', lw=0.5)
    plt.setp(ax[k], title=str(obj2_allramppads_shifted[k].shape))
ax[5].imshow(np.angle(obj_GT), extent=[.0, obj_GT.shape[0], .0, obj_GT.shape[1]])
ax[5].axvline(x=obj_GT.shape[1]/2, c='r', lw=0.5)
ax[5].axhline(y=obj_GT.shape[0]/2, c='r', lw=0.5)
plt.setp(ax[5], title=str(obj_GT.shape))

# after cropping them
fig, ax = plt.subplots(nrow, ncol, figsize=(3*ncol, 3*nrow))
plt.tight_layout()
for k in range(ncol-1):
    ax[k].imshow(np.angle(obj2_alliter[k]), extent=[.0, obj2_alliter[k].shape[0], .0, obj2_alliter[k].shape[1]])
    ax[k].axvline(x=obj2_alliter[k].shape[1]/2, c='k', lw=0.5)
    ax[k].axhline(y=obj2_alliter[k].shape[0]/2, c='k', lw=0.5)
    plt.setp(ax[k], title=str(obj2_alliter[k].shape))
ax[5].imshow(np.angle(obj_GT), extent=[.0, obj_GT.shape[0], .0, obj_GT.shape[1]])
ax[5].axvline(x=obj_GT.shape[1]/2, c='r', lw=0.5)
ax[5].axhline(y=obj_GT.shape[0]/2, c='r', lw=0.5)
plt.setp(ax[5], title=str(obj_GT.shape))

# ai solution
fig, ax = plt.subplots(nrow, ncol, figsize=(3*ncol, 3*nrow))
plt.tight_layout()
for k in range(ncol-1):
    ax[k].imshow(np.angle(cropped_arrays[k]), extent=[.0, cropped_arrays[k].shape[0], .0, cropped_arrays[k].shape[1]])
    ax[k].axvline(x=cropped_arrays[k].shape[1]/2, c='k', lw=0.5)
    ax[k].axhline(y=cropped_arrays[k].shape[0]/2, c='k', lw=0.5)
    plt.setp(ax[k], title=str(cropped_arrays[k].shape))
ax[5].imshow(np.angle(obj_GT), extent=[.0, obj_GT.shape[0], .0, obj_GT.shape[1]])
ax[5].axvline(x=obj_GT.shape[1]/2, c='r', lw=0.5)
ax[5].axhline(y=obj_GT.shape[0]/2, c='r', lw=0.5)
plt.setp(ax[5], title=str(obj_GT.shape))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

type: name 'obj2_allramps' is not defined

In [3]:
%timeit LD40 = livedifferences.LoadData(offline_fname=ol_cases['40px']['offline_fname'], realtime_fname=ol_cases['40px']['realtime_fname'], samplename=ol_cases['40px']['samplename'])
#719 ns ± 0.659 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)
LD40 = livedifferences.LoadData(offline_fname=ol_cases['40px']['offline_fname'], realtime_fname=ol_cases['40px']['realtime_fname'], samplename=ol_cases['40px']['samplename'])

729 ns ± 0.824 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [4]:
%timeit LD40.load()
#128 ms ± 205 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)

128 ms ± 254 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
%timeit LD40.process_data() # Gives obj1_abs, obj2_abs, obj1_abslog, obj2_abslog, obj1_phase, obj2_phase, w1, w2, obj1_ramp, ramp1, obj2_ramp, ramp2, obj1_phaseramp, obj2_phaseramp, obj1_phaseramp_unwrapped, obj2_phaseramp_unwrapped, obj1_rmramp_abs, obj2_rmramp_abs, obj1_rmramp_abslog, obj2_rmramp_abslog, diff_rmramp_abslog, diff_abslog_od, diff_phase, diff_phaseramp, diff_phaseramp_unwrapped, diff_phaseramp_unwrapped_ramp, shift1, error, phasediff
#1.91 s ± 13.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
#1.89 s ± 5.23 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
# removed phase_cross_correlation calculation: 1.58 s ± 10.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
# removed the hole try-except part: 1.3 s ± 6.27 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

1.3 s ± 6.27 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
%timeit LD40.load_GT() # Gives fname3, fname_pr, pr, obj, obj_abs, obj_abslog, obj_phase, w, obj_ramp, ramp, obj_rmramp_abs, obj_rmramp_abslog, obj_phaseramp, obj_phaseramp_unwrapped

998 ms ± 10.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%timeit LD40.add_padding() # Gives sh, sh1, padrow, padcol, obj1_pad, obj2_pad, obj1_abs_pad, obj2_abs_pad, obj1_abslog_pad, obj2_abslog_pad, obj1_phase_pad, obj2_phase_pad, obj1_ramp_pad, obj2_ramp_pad, obj1_rmramp_abs_pad, obj2_rmramp_abs_pad, obj1_rmramp_abslog_pad, obj2_rmramp_abslog_pad, obj1_phaseramp_pad, obj2_phaseramp_pad, obj1_phaseramp_unwrapped_pad, obj2_phaseramp_unwrapped_pad

47.3 ms ± 544 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
